In [1]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, cc, fci
from flax import linen as nn
import flax.nnx as nnx
import optax
from tqdm import tqdm
import time 
from functools import partial
from jax import flatten_util
import sys
sys.path.append('..')
from VMC_tool import SingleStateAnsatz,create_machine,compute_local_energies,\
    compute_qgt,forces_expect_hermitian

/opt/miniconda3/envs/Netket/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


∣NK⟩ Tip: You can disable these tips by setting export NETKET_NO_TIPS=1 in your .bashrc.

In [9]:
# ====================== 1. 原有分子 & SCF 部分（复用你的代码） ======================
bond_length = 1.595
geometry = [('Li', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()

n_electrons = mol.nelectron
n_orbitals = mol.nao_nr()
n_spin_orbitals = 2 * n_orbitals

# RHF 下 占据轨道数、虚轨道数
n_occ = mf.mol.nelectron // 2    # 闭壳层，α/β 占据轨道数相同
n_vir = n_orbitals - n_occ
print(f"实轨道数: {n_orbitals}, 占据轨道数: {n_occ}, 虚轨道数: {n_vir}")

# ====================== 2. PySCF 运行 CCSD，提取激发振幅 ======================
ccsd = cc.CCSD(mf)
ccsd.run(verbose=0)

# t1: 单激发振幅 shape = (n_occ, n_vir)
# t1[i,a] : 占据轨道i → 虚轨道a 的单激发振幅
t1 = ccsd.t1
# 阈值：筛选显著跃迁（可根据体系调小/调大）
thresh = 1e-3

# 收集【实轨道层面】的跃迁对 (occ_orb, vir_orb)
orbital_trans_pairs = []
for i in range(n_occ):
    for a in range(n_vir):
        amp = abs(t1[i, a])
        if amp > thresh:
            # 虚轨道全局编号 = n_occ + a
            occ = i
            vir = n_occ + a
            orbital_trans_pairs.append((occ, vir))

print(f"\nCCSD 筛选出的实轨道跃迁对（单激发）: {orbital_trans_pairs}")

# ====================== 3. 实轨道对 → 转换为 NetKet 自旋轨道 edges ======================
# 规则：
# 实轨道 o → α自旋轨道: 2*o
# 实轨道 o → β自旋轨道: 2*o + 1
edges = []
for o_occ, o_vir in orbital_trans_pairs:
    # α 自旋跃迁
    so_alpha_occ = 2 * o_occ
    so_alpha_vir = 2 * o_vir
    edges.append((so_alpha_occ, so_alpha_vir))
    
    # β 自旋跃迁（闭壳层对称）
    so_beta_occ = 2 * o_occ + 1
    so_beta_vir = 2 * o_vir + 1
    edges.append((so_beta_occ, so_beta_vir))

print(f"\n转换后 NetKet 可用自旋轨道 edges:\n{edges}")

# ====================== 4. 后续构建 Hilbert 空间 & 采样器（完全对接你原有逻辑） ======================
n_fermions_per_spin = (n_electrons//2, n_electrons//2)
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=n_orbitals,
    s=1/2,
    n_fermions_per_spin=n_fermions_per_spin,
)
ha = nkx.operator.from_pyscf_molecule(mol)

# 用 CCSD 生成的 edges 构建图与跃迁规则
g = nk.graph.Graph(edges=edges)
single_rule = nk.sampler.rules.FermionHopRule(hilbert=hi, graph=g)
sampler = nk.sampler.MetropolisSampler(hi, rule=single_rule, n_chains=100, sweep_size=32)

print(f"\nHilbert 空间维度: {hi.size}")
print("已基于 CCSD 跃迁路径构建 MCMC 采样器")

实轨道数: 6, 占据轨道数: 2, 虚轨道数: 4

CCSD 筛选出的实轨道跃迁对（单激发）: [(1, 2), (1, 5)]

转换后 NetKet 可用自旋轨道 edges:
[(2, 4), (3, 5), (2, 10), (3, 11)]

Hilbert 空间维度: 12
已基于 CCSD 跃迁路径构建 MCMC 采样器


In [3]:
import jax
import jax.numpy as jnp
from functools import partial

# ==============================================
# 1. 生成随机初始态
# ==============================================
def generate_random_initial_states(hi, n_chains: int, seed: int = 42):
    key = jax.random.PRNGKey(seed)
    keys = jax.random.split(key, n_chains)
    return jax.vmap(lambda k: hi.random_state(k))(keys)

# ==============================================
# 2. 候选状态生成
# ==============================================
def make_get_all_next_states(edges):
    @jax.jit
    def get_all_next_states_jit(S: jnp.ndarray):
        next_states = []
        valid_masks = []
        for (i, j) in edges:
            occ_i = S[..., i]
            occ_j = S[..., j]
            valid = (occ_i != occ_j)
            new_state = S.at[..., i].set(occ_j).at[..., j].set(occ_i)
            next_states.append(new_state)
            valid_masks.append(valid)
        return jnp.stack(next_states), jnp.stack(valid_masks)
    return get_all_next_states_jit

# ==============================================
# 3. Metropolis 单步跃迁
# ==============================================
def make_metropolis_hastings_step(edges, machine):
    get_all_next = make_get_all_next_states(edges)
    
    @jax.jit
    def mh_step(params, state: jnp.ndarray, key: jax.Array):
        candidates, valid_mask = get_all_next(state[None, :])
        candidates = candidates[:, 0]
        valid_mask = valid_mask[:, 0]
        
        key, subk = jax.random.split(key)
        idx = jax.random.choice(subk, len(edges))
        cand = candidates[idx]
        is_valid = valid_mask[idx]
        
        log_curr = machine(params, state)
        log_cand = machine(params, cand)
        log_acc = 2 * jnp.real(log_cand - log_curr)
        
        key, subk = jax.random.split(key)
        accept = is_valid & (log_acc > jnp.log(jax.random.uniform(subk)))
        new_state = jnp.where(accept, cand, state)
        return new_state, key
    
    return mh_step

# ==============================================
# 🔥 最终版：带 sampler_state + 随机数管理 + 对齐 NetKet
# ==============================================
@partial(jax.jit, static_argnums=(0,1,3,4,6))
def mcmc_sampler_multichain(
    n_samples_per_chain: int,
    n_warmup: int,             # 单位：sweep
    sampler_state: tuple,      # ✅ NetKet 风格状态：(current_states, chain_keys)
    edges: tuple,
    machine: callable,
    params: dict,
    sweep_size: int = 32       # ✅ 保留 sweep_size
):
    # 解开 sampler_state（和 NetKet 完全一致）
    current_states, current_keys = sampler_state
    n_chains = current_states.shape[0]
    mh_step = make_metropolis_hastings_step(edges, machine)

    # -------------------------
    # 一次 sweep = 连续跳 sweep_size 次
    # -------------------------
    def single_sweep(carry, _):
        states, keys = carry
        # 多链并行 VMAP
        (new_s, new_k), _ = jax.lax.scan(
            lambda c, _: (jax.vmap(mh_step, in_axes=(None, 0, 0))(params, c[0], c[1]), None),
            (states, keys),
            length=sweep_size
        )
        return (new_s, new_k), new_s

    # -------------------------
    # 1) Warmup（仅更新状态，不保存样本）
    # -------------------------
    if n_warmup > 0:
        (current_states, current_keys), _ = jax.lax.scan(
            single_sweep, (current_states, current_keys), length=n_warmup
        )

    # -------------------------
    # 2) 正式采样（保存样本 + 更新最终状态）
    # -------------------------
    (final_states, final_keys), samples = jax.lax.scan(
        single_sweep, (current_states, current_keys), length=n_samples_per_chain
    )

    # 打包新的 sampler_state（返回给下一次迭代）
    new_sampler_state = (final_states, final_keys)
    
    # 展平样本：[n_samples, n_chains, n_sites] → [n_samples*n_chains, n_sites]
    samples_flat = samples.reshape(-1, current_states.shape[-1])
    return samples_flat, new_sampler_state

In [4]:
# 初始化链状态 + 随机数状态（构成 sampler_state）
def init_sampler_state(hi, n_chains, seed=42):
    init_states = generate_random_initial_states(hi, n_chains, seed)
    key = jax.random.PRNGKey(seed)
    chain_keys = jax.random.split(key, n_chains)  # 每条链独立随机数
    return (init_states, chain_keys)

In [5]:
# ======================
# 超参数
# ======================
N_CHAINS = 16
N_WARMUP = 32
N_SAMPLES_PER_CHAIN = 100
SWEEP_SIZE = 32

# ======================
# 初始化 ONCE
# ======================
rngs = nnx.Rngs(21)
model = SingleStateAnsatz(hi.size, hidden_dim=16, rngs=rngs)
machine, graphdef, params = create_machine(model)

sampler_state = init_sampler_state(hi, N_CHAINS, seed=42)
samples, sampler_state = mcmc_sampler_multichain(
    n_samples_per_chain=N_SAMPLES_PER_CHAIN,
    n_warmup=N_WARMUP,
    sampler_state=sampler_state,  # ✅ 状态传递
    edges=((0,1),(2,3)),
    machine=machine,
    params=params,
    sweep_size=SWEEP_SIZE
)
samples.shape

(1600, 12)

In [11]:
import jax
import jax.numpy as jnp
import optax
import time
from functools import partial
import flax.nnx as nnx

# ===================== 6. 初始化（适配多链） =====================
rngs = nnx.Rngs(21)
model = SingleStateAnsatz(hi.size, hidden_dim=16, rngs=rngs)
machine, graphdef, params = create_machine(model)

optimizer = optax.sgd(learning_rate=0.001)
opt_state = optimizer.init(params)

# 推荐参数（和 NetKet 一样快、一样准）
N_CHAINS = 100
N_SAMPLES_PER_CHAIN = 20
N_WARMUP = 10
SWEEP_SIZE = 32
N_ITER = 300

# ===================== ✅ 关键：初始化 sampler_state（只初始化一次！）=====================
def init_sampler_state(hi, n_chains, seed=42):
    init_states = generate_random_initial_states(hi, n_chains, seed)
    key = jax.random.PRNGKey(seed)
    chain_keys = jax.random.split(key, n_chains)
    return (init_states, chain_keys)

# 初始化一次，后面永远复用、更新
sampler_state = init_sampler_state(hi, N_CHAINS, seed=21)

# ===================== 7. 训练循环（✅ 完全修复版）=====================
print("\n" + "="*60)
print("开始多链 VMC 训练 (自然梯度下降法)")
print("="*60)

history = {
    'step': [],
    'energy': [],
    'energy_std': [],
    'error': []
}
start_time = time.time()

for step in range(N_ITER):
    # ======================================================================
    # ✅ 1. 采样：复用 sampler_state，不每次重新生成初始态！（核心修复）
    # ======================================================================
    samples, sampler_state = mcmc_sampler_multichain(
        n_samples_per_chain=N_SAMPLES_PER_CHAIN,
        n_warmup=N_WARMUP,
        sampler_state=sampler_state,  # 状态传递
        edges=((0,1),(2,3)),
        machine=machine,
        params=params,
        sweep_size=SWEEP_SIZE
    )

    # ======================================================================
    # 2. 能量 & 自然梯度（不变）
    # ======================================================================
    energy, energy_std, grad = forces_expect_hermitian(machine, params,ha, samples)
    grad = jax.tree_util.tree_map(lambda x: x * 2, grad)

    qgt_reg, qgt_unravel_fun = compute_qgt(machine, params, samples, diag_shift=0.001)
    grad_flat, grad_unravel_fn = flatten_util.ravel_pytree(grad)
    natural_grad_flat = jnp.linalg.solve(qgt_reg, grad_flat)
    natural_grad = grad_unravel_fn(natural_grad_flat)

    # ======================================================================
    # 3. 参数更新
    # ======================================================================
    updates, opt_state = optimizer.update(natural_grad, opt_state, params)
    params = optax.apply_updates(params, updates)

    # ======================================================================
    # 4. 日志
    # ======================================================================
    if step % 1 == 0 or step == N_ITER - 1:
        error = jnp.abs(energy.real - E_fcis[0])
        history['step'].append(step)
        history['energy'].append(float(energy.real))
        history['energy_std'].append(float(energy_std))
        history['error'].append(float(error))
        print(f"Step {step:3d} | E: {energy.real:.8f} ± {energy_std:.6f} | FCI: {E_fcis[0]:.8f} | Error: {error:.6f}")

end_time = time.time()
print(f"\n训练耗时：{end_time - start_time:.2f} 秒")

# ======================================================================
# 最终能量评估
# ======================================================================
final_samples, _ = mcmc_sampler_multichain(
    n_samples_per_chain=N_SAMPLES_PER_CHAIN * 2,
    n_warmup=5,
    sampler_state=sampler_state,
    edges=((0,1),(2,3)),
    machine=machine,
    params=params,
    sweep_size=SWEEP_SIZE
)
final_energy, final_std, _ = forces_expect_hermitian(machine, params, final_samples)
final_error = jnp.abs(final_energy.real - E_fcis[0])

print("\n" + "="*60)
print(f"训练完成!")
print(f"最终能量：{final_energy.real:.8f} ± {final_std:.6f} Ha")
print(f"FCI 基准：{E_fcis[0]:.8f} Ha")
print(f"绝对误差：{final_error:.6f} Ha")
print(f"相对误差：{final_error / jnp.abs(E_fcis[0]) * 100:.4f}%")
print("="*60)


开始多链 VMC 训练 (自然梯度下降法)
Step   0 | E: -3.68664305 ± 0.044492 | FCI: -7.88240193 | Error: 4.195759
Step   1 | E: -3.24271939 ± 0.541419 | FCI: -7.88240193 | Error: 4.639683
Step   2 | E: -63.57300998 ± 30.544874 | FCI: -7.88240193 | Error: 55.690608
Step   3 | E: -99091508370015.17187500 ± 18402248904367.261719 | FCI: -7.88240193 | Error: 99091508370007.296875
Step   4 | E: nan ± nan | FCI: -7.88240193 | Error: nan
Step   5 | E: nan ± nan | FCI: -7.88240193 | Error: nan
Step   6 | E: nan ± nan | FCI: -7.88240193 | Error: nan
Step   7 | E: nan ± nan | FCI: -7.88240193 | Error: nan
Step   8 | E: nan ± nan | FCI: -7.88240193 | Error: nan
Step   9 | E: nan ± nan | FCI: -7.88240193 | Error: nan
Step  10 | E: nan ± nan | FCI: -7.88240193 | Error: nan
Step  11 | E: nan ± nan | FCI: -7.88240193 | Error: nan
Step  12 | E: nan ± nan | FCI: -7.88240193 | Error: nan


KeyboardInterrupt: 

In [ ]:
jax.__version__